In [1]:
import cv2

frame = cv2.imread('./data/balloons.jpg')

# Grayscale
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# Resize
resized = cv2.resize(gray, (224, 224))
normalized = resized / 255.0


print(normalized.shape)
print(normalized.min(), normalized.max())


(224, 224, 3)
0.0 1.0


In [2]:
import torch

tensor = torch.tensor(normalized, dtype=torch.float32)
tensor = tensor.permute(2, 0, 1)   # (H, W, 3) → (3, H, W)
tensor = tensor.unsqueeze(0)        # (3, H, W) → (1, 3, H, W)

print(tensor.shape)  # torch.Size([1, 3, 224, 224])

torch.Size([1, 3, 224, 224])


In [ ]:
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, state_size=4, num_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_actions),
        )

    def forward(self, x):
        return self.net(x)

model = DQN()

torch.Size([1, 5])


In [7]:
import gymnasium as gym
env = gym.make("CartPole-v1")
state, _ = env.reset()
print(state)

[ 0.00297961 -0.01695436 -0.01979636  0.01107473]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# Hyperparameters
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.01
gamma = 0.99          # discount factor
batch_size = 32
lr = 1e-4

model = DQN(num_actions=5)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.MSELoss()
memory = deque(maxlen=10000)  # replay buffer

def select_action(state):
    if random.random() < epsilon:
        return random.randint(0, 4)
    with torch.no_grad():
        return model(state).argmax().item()

def train():
    if len(memory) < batch_size:
        return

    batch = random.sample(memory, batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    states      = torch.cat(states)
    next_states = torch.cat(next_states)
    actions     = torch.tensor(actions)
    rewards     = torch.tensor(rewards, dtype=torch.float32)
    dones       = torch.tensor(dones, dtype=torch.float32)

    current_q   = model(states).gather(1, actions.unsqueeze(1)).squeeze()
    next_q      = model(next_states).max(1).values
    target_q    = rewards + gamma * next_q * (1 - dones)

    loss = loss_fn(current_q, target_q.detach())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Training loop
for episode in range(1000):
    state = env.reset()
    done = False

    while not done:
        action = select_action(state)
        next_state, reward, done, _ = env.step(action)

        memory.append((state, action, reward, next_state, done))
        train()
        state = next_state

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

tensor(1., grad_fn=<SumBackward0>)
